## SynthStrip: Deep Learning–Based Skull-Stripping for MRI

**SynthStrip** is a state-of-the-art, deep learning–based skull-stripping tool designed to remove non-brain tissues (skull, scalp, eyes, and background) from MRI scans. Developed by the Martinos Center (MIT/MGH) as part of the FreeSurfer ecosystem, SynthStrip is built to be **modality-agnostic**, meaning it works reliably on T1-, T2-, FLAIR-, PD-weighted, and even unconventional MRI contrasts.

### How SynthStrip Works
SynthStrip uses a U-Net–like convolutional neural network trained entirely on **synthetic MRI volumes**. Synthetic data allows perfect ground-truth brain masks and diverse variations in image contrast, noise, resolution, and artifacts. This training strategy eliminates the limitations of traditional skull-stripping tools that depend heavily on specific MRI modalities or scanner parameters.

### Advantages
- **Robust across all MRI types** (T1, T2, FLAIR, etc.)
- **Resistant to noise, artifacts, and orientation variations**
- **Requires no modality-specific tuning**
- **Produces high-quality brain masks suitable for clinical and deep-learning pipelines**

### Purpose in Preprocessing
SynthStrip is commonly used as an early preprocessing step in neuroimaging pipelines to:
- isolate brain tissue,
- remove irrelevant anatomical structures,
- improve downstream tasks such as tumor segmentation, classification, and registration.

https://surfer.nmr.mgh.harvard.edu/docs/synthstrip/


Setup Docker and SynthStrip

In [ ]:
# Install Docker
!apt-get update -qq
!apt-get install -y docker.io -qq
!service docker start

# Download SynthStrip wrapper and pull Docker image
!curl -s -O https://raw.githubusercontent.com/freesurfer/freesurfer/dev/mri_synthstrip/synthstrip-docker
!chmod +x synthstrip-docker
!docker pull freesurfer/synthstrip:latest

# Install Python libraries
!pip install nibabel opencv-python -q

In [2]:
import os
INPUT_FOLDER = '/content/drive/MyDrive/Semestre 3/IMAD/Mini projet/DS DL cleaning/ds'
OUTPUT_FOLDER = '/content/drive/MyDrive/Semestre 3/IMAD/Mini projet/DS DL cleaning/brain_tumor_stripped'

# Create output folder
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print(f"Input: {INPUT_FOLDER}")
print(f"Output: {OUTPUT_FOLDER}\n")

Input: /content/drive/MyDrive/Semestre 3/IMAD/Mini projet/DS DL cleaning/ds
Output: /content/drive/MyDrive/Semestre 3/IMAD/Mini projet/DS DL cleaning/brain_tumor_stripped



## Process All Images

In [ ]:
import glob
import cv2
import numpy as np
import nibabel as nib
import subprocess
from tqdm.notebook import tqdm

# Find all JPG files
jpg_files = glob.glob(f'{INPUT_FOLDER}/**/*.jpg', recursive=True) + \
            glob.glob(f'{INPUT_FOLDER}/**/*.jpeg', recursive=True)

print(f"Found {len(jpg_files)} images\n")

# Create temp folders
os.makedirs('/content/temp_input', exist_ok=True)
os.makedirs('/content/temp_output', exist_ok=True)

# Process each image
success = 0
for jpg_file in tqdm(jpg_files, desc="Processing"):
    try:
        # Get output path (keeps folder structure)
        rel_path = os.path.relpath(jpg_file, INPUT_FOLDER)
        output_file = os.path.join(OUTPUT_FOLDER, rel_path)
        os.makedirs(os.path.dirname(output_file), exist_ok=True)

        # Skip if already done
        if os.path.exists(output_file):
            success += 1
            continue

        # Convert JPG to NIfTI
        basename = os.path.splitext(os.path.basename(jpg_file))[0]
        nii_input = f'/content/temp_input/{basename}.nii.gz'
        nii_output = f'/content/temp_output/{basename}.nii.gz'

        img = cv2.imread(jpg_file, cv2.IMREAD_GRAYSCALE)
        img_3d = np.expand_dims(img, axis=2)
        nib.save(nib.Nifti1Image(img_3d.astype(np.float32), np.eye(4)), nii_input)

        # Run SynthStrip
        result = subprocess.run(
            ['./synthstrip-docker', '-i', nii_input, '-o', nii_output],
            capture_output=True,
            timeout=120
        )

        if result.returncode == 0:
            # Convert back to JPG
            data = nib.load(nii_output).get_fdata()[:, :, 0]
            data = ((data - data.min()) / (data.max() - data.min() + 1e-8) * 255).astype(np.uint8)
            cv2.imwrite(output_file, data)
            success += 1

    except Exception as e:
        print(f"\nError: {os.path.basename(jpg_file)} - {str(e)}")

# Summary
print(f"\n{'='*50}")
print(f"✓ Processed: {success}/{len(jpg_files)} images")
print(f"✓ Output: {OUTPUT_FOLDER}")
print(f"{'='*50}")